In [2]:
# A100 setup
import subprocess, sys, zipfile
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'peft==0.15.2', 'accelerate==1.6.0', 'scipy==1.15.3'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'bitsandbytes'])
with zipfile.ZipFile('/content/controlled_editing_code_a100.zip') as z:
    z.extractall('/content')
import torch
print('GPU:', torch.cuda.get_device_name(0))
assert 'A100' in torch.cuda.get_device_name(0)
print('A100 SOURCE READY')


GPU: NVIDIA A100-SXM4-40GB
A100 SOURCE READY


In [3]:
# Three independent A100 seeds; fixed three epochs each.
import subprocess, sys, shutil, json
from pathlib import Path
root = Path('/content/cvpr2027-a100'); root.mkdir(exist_ok=True)
for seed in (17, 29, 41):
    run = root / f'seed-{seed}'
    if (run/'run_manifest.json').exists() and json.loads((run/'run_manifest.json').read_text()).get('status') == 'completed':
        continue
    with (root/f'seed-{seed}.log').open('a') as log:
        process = subprocess.Popen([sys.executable, '-u', '/content/train_controlled_editing.py', '--output', str(run), '--epochs', '3', '--seed', str(seed), '--resume'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait()
    assert code == 0, f'Seed {seed} failed with exit code {code}'
    print('COMPLETED SEED', seed, flush=True)
shutil.make_archive('/content/cvpr2027-a100', 'zip', root)
print('ALL THREE SEEDS COMPLETE; archive ready')


2026-09-13 06:00:08.209371: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-13 06:00:08.272011: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag 

In [1]:
# Download the completed A100 experiment to the local CVPR folder.
from google.colab import files
files.download('/content/cvpr2027-a100.zip')


FileNotFoundError: Cannot find file: /content/cvpr2027-a100.zip